# Missing values, types and duplicates

Python stage, lesson P16. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** count missing values and test for them with `isna`; choose between dropping, filling and keeping a gap, and say what each hides; read columns with the type you want, whole numbers that can be missing and codes as text; find and remove duplicates on a stated key.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it.

## Setup

In [ ]:
import math
import numbers

import io

import numpy as np
import pandas as pd

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
EXERCISES = ["Exercise 1", "Exercise 2", "Exercise 3"]


RESULTS = {}


def same(got, expected):
    """True when two values agree: floats within rounding, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k]) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong):
            print("Hint:", message)
    return False

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.1.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/"
stores = pd.read_csv(BASE + "stores.csv")
customers = pd.read_csv(BASE + "customers.csv")
products = pd.read_csv(BASE + "products.csv")
orders = pd.read_csv(BASE + "orders.csv")
order_items = pd.read_csv(BASE + "order_items.csv")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
assert (len(stores), len(customers), len(products), len(orders), len(order_items)) == (3, 150, 16, 1800, 3287)
print("Data loaded: 3 stores, 150 customers, 16 products, 1,800 orders, 3,287 order lines.")
# The order lines with their price and amount, as built in lesson P13.
price_of = dict(zip(products["product_id"], products["unit_price_usd"]))
lines = order_items.assign(price=order_items["product_id"].map(price_of))
lines = lines.assign(amount=lines["quantity"] * lines["price"] - lines["discount_usd"])

# Exercise 2 uses this table: the first eight customers as a signup desk typed them, two of them twice.
signup_log = customers.head(8).loc[[0, 1, 2, 2, 3, 4, 5, 5, 6, 7]].reset_index(drop=True)
signup_log.loc[3, "loyalty_tier"] = "silver"  # customer 3, typed again with another tier

## Unit 1: isna: finding the gaps

**Predict first.** which columns of `orders` have gaps, and how many? On the last line, how many rows will the test `== np.nan` find, and how many will `isna()` find?

In [ ]:
print(orders.isna().sum())
print((orders["customer_id"] == np.nan).sum(), orders["customer_id"].isna().sum())

**Predict first.** does `np.nan` equal itself? And what does `isna` say for NaN, `None`, `NaT` and `pd.NA`?

In [ ]:
print(np.nan == np.nan)
print(pd.isna(np.nan), pd.isna(None), pd.isna(pd.NaT), pd.isna(pd.NA))

## Unit 2: dropna and fillna: choose, then count

**Predict first.** how many rows will `kept` and `coded` have, and how much money did the dropped order carry?

In [ ]:
kept = orders.dropna(subset=["customer_id"])
coded = orders.assign(customer_id=orders["customer_id"].fillna(0))
print(len(orders), len(kept), len(coded))
print(lines.loc[lines["order_id"] == 1335, "amount"].sum())

**Predict first.** how many different customers are there before the fill, and after it?

In [ ]:
print(orders["customer_id"].nunique(), coded["customer_id"].nunique())

## Unit 3: dtype: read the type you want

**Predict first.** customer ids are whole numbers, and one is missing. How do the first three print as read, and with `dtype="Int64"`?

In [ ]:
as_read = orders["customer_id"].head(3).tolist()
whole = pd.read_csv(BASE + "orders.csv", dtype={"customer_id": "Int64"})
print(as_read, whole["customer_id"].head(3).tolist(), whole["customer_id"].isna().sum())

**Predict first.** a loyalty card number is a code, not a quantity. What does each read give?

In [ ]:
card_file = "card_no,tier\n0042,gold\n0107,silver\n0003,bronze\n"
as_number = pd.read_csv(io.StringIO(card_file))
as_text = pd.read_csv(io.StringIO(card_file), dtype={"card_no": "string"})
print(as_number["card_no"].tolist(), as_text["card_no"].tolist())

## Unit 4: duplicates: say what makes a row the same

**Predict first.** how many repeated orders does `orders` hold? In `exact`, order 1003 is typed twice: which of the six rows does `duplicated()` mark?

In [ ]:
print(orders.duplicated().sum())
exact = orders.head(5).loc[[0, 1, 2, 2, 3, 4]]
print(exact.duplicated().tolist())

**Predict first.** order 1003 is typed twice again, as mobile and then as cash. How many rows does `drop_duplicates()` leave, and how many with `subset=["order_id"]`?

In [ ]:
near = exact.assign(payment_method=["mobile", "mobile", "mobile", "cash", "mobile", "mobile"])
print(len(near.drop_duplicates()), len(near.drop_duplicates(subset=["order_id"])))

## Unit 5: Check: a cleaning log

**Predict first.** how many rows are left after each step?

In [ ]:
log = [("start", len(orders))]
step = orders.drop_duplicates(subset=["order_id"])
log.append(("drop_duplicates", len(step)))
step = step.dropna(subset=["customer_id"])
log.append(("dropna", len(step)))
print(log)

## Finish it

### Exercise 1: Finish it

**Task:** What are the three store codes of the till's store list, exactly as written? The list is a small file typed here. Replace the first `None` with the file read with `read_csv`, with the `store_code` column kept as text, then set `answer_1` to the list of that column.

**Expected:** a list of three codes, as text.

In [ ]:
store_file = "store_code,store_name\n01,Old Town\n02,Campus\n03,Harbor\n"
store_codes = None  # replace None: store_file read with pd.read_csv(io.StringIO(store_file)), with store_code kept as text
answer_1 = None  # then set answer_1 to the list of the store_code column

In [ ]:
_ = check("Exercise 1", "answer_1", ["01", "02", "03"], traps={(1, 2, 3): "[1, 2, 3] are numbers: the zeros are gone. Read store_code as text, with the dtype parameter of read_csv."})

<details>
<summary>Hint</summary>

Unit 3 reads a code as text with the `dtype` parameter of `read_csv`, a dictionary from column name to type.

</details>

<details>
<summary>Solution</summary>

```python
store_file = "store_code,store_name\n01,Old Town\n02,Campus\n03,Harbor\n"
store_codes = pd.read_csv(io.StringIO(store_file), dtype={"store_code": "string"})
answer_1 = store_codes["store_code"].tolist()
```

The answer is `['01', '02', '03']`. Read without `dtype`, the codes become the numbers 1, 2 and 3, and the check cell says so.

</details>

## On your own

### Exercise 2: On your own

**Task:** How many different customers does the signup log hold? `signup_log` is the first eight customers as a signup desk typed them, and the setup cell of the notebook makes it. Customer 6 was typed twice in the same way, and customer 3 twice with two different tiers. A customer is the same customer when the `customer_id` is the same. Before you start, say in one sentence how many rows you expect to keep, and why. Write the code in the empty cell below and set `answer_2`.

**Expected:** one whole number.

In [ ]:
_ = check("Exercise 2", "answer_2", 8, traps={9: "9 means customer 3 was kept twice: the two typings have different tiers, so a plain drop_duplicates() keeps both. Name the key.", 10: "10 is the number of rows of the log. Remove the repeated customers."})

<details>
<summary>Hint</summary>

Unit 4 removes repeated rows. A repeat that differs in one column is not seen as a repeat unless the key is named.

</details>

<details>
<summary>Solution</summary>

```python
answer_2 = len(signup_log.drop_duplicates(subset=["customer_id"]))
```

The answer is 8: ten rows, two of them repeated customers, 3 and 6. `drop_duplicates()` without a key removes only the exact repeat of customer 6 and leaves 9, and the check cell says so.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague counts the Alameda customers who were referred by another customer, and the answer looks too large. Fix the line, then set `answer_3 = len(referred)`.

**Expected:** one whole number.

In [ ]:
alameda = customers[customers["city"] == "Alameda"]
referred = alameda[alameda["referred_by"] != np.nan]  # the colleague's line
answer_3 = None  # fix the line above, then set answer_3 = len(referred)

In [ ]:
_ = check("Exercise 3", "answer_3", 10, traps={60: "60 is every Alameda customer: != np.nan is True for every row, gaps included. Test for the gap with the method of unit 1."})

<details>
<summary>Hint</summary>

Print `alameda["referred_by"] != np.nan` and look at it. A gap is not equal to anything, so the test is `True` for it too.

</details>

<details>
<summary>Solution</summary>

```python
alameda = customers[customers["city"] == "Alameda"]
referred = alameda[alameda["referred_by"].notna()]
answer_3 = len(referred)
```

The answer is 10. The colleague's test said `True` for all 60 Alameda customers, because a gap is not equal to anything, so `!=` is true for it. `notna()` asks the right question, and the check cell says so.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")